# Lenta autoresearch — Kaggle experiment notebook

Agent modifies KNOBS below to maximize `main_metric` on subset (26_12-20, 71 GT).
Baseline: RapidOCR + UPSCALE 3 + YOLO conf 0.1 + FPS 3.

In [ ]:
# === KNOBS (agent edits these) ===
OCR_ENGINE = 'rapidocr'
UPSCALE = 3.0
USE_SHARPEN = False
USE_CLAHE = False
USE_DENOISE = False
YOLO_CONF = 0.10
BBOX_PADDING = 0.25
FPS_SAMPLE = 3
DEFAULT_NET_FOR_TEXT_FIELDS = False
USE_QR = True
DESCRIPTION = 'v11: barcode-zone crops (full/bot50/botright/right50) x {1x,2x,4x} x {raw,otsu,adapt}, top-8 frames'
# === END KNOBS ===
print(f'DESCRIPTION: {DESCRIPTION}')
print(f'OCR={OCR_ENGINE} UPSCALE={UPSCALE} SHARPEN={USE_SHARPEN} CLAHE={USE_CLAHE} DENOISE={USE_DENOISE}')
print(f'YOLO_CONF={YOLO_CONF} BBOX_PADDING={BBOX_PADDING} FPS_SAMPLE={FPS_SAMPLE} DEFAULT_NET={DEFAULT_NET_FOR_TEXT_FIELDS}')

In [ ]:
# === Install (torch 2.4 ALWAYS — P100 kernels) ===
import subprocess, sys
print('=== Installing torch 2.4 (P100 compat)... ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.4.1', 'torchvision==0.19.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)

extras = ['ultralytics', 'rapidocr-onnxruntime', 'rapidfuzz', 'pytesseract',
          'pyzbar', 'zxing-cpp', 'tqdm', 'huggingface-hub']
if OCR_ENGINE == 'easyocr': extras.append('easyocr')
if OCR_ENGINE == 'qwen25_vl_3b':
    extras += ['transformers==4.49.0', 'qwen-vl-utils', 'accelerate']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + extras, check=True)

apt_pkgs = 'libzbar0'
if OCR_ENGINE == 'tesseract':
    apt_pkgs += ' tesseract-ocr tesseract-ocr-rus tesseract-ocr-eng'
subprocess.run(f'apt-get install -y -q {apt_pkgs} 2>/dev/null', shell=True)

import torch
print(f'torch {torch.__version__}, CUDA {torch.version.cuda}, GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# === Data discovery ===
import os, cv2, pandas as pd
video_path = csv_path = None
for root,_,files in os.walk('/kaggle/input'):
    for f in files:
        if f == '26_12-20.mp4' and os.path.basename(root) == '26_12-20':
            video_path = os.path.join(root, f)
            csv_path = os.path.join(root, '26_12-20.csv')
print(f'video: {video_path}, csv exists: {os.path.exists(csv_path) if csv_path else False}')
assert video_path and csv_path and os.path.exists(csv_path), 'Subset data not found'

In [ ]:
# === Clone repo (autoresearch/may16 branch) + download YOLO weights ===
import subprocess
from pathlib import Path
subprocess.run('rm -rf /kaggle/working/lenta && git clone --depth 1 --branch autoresearch/may16 https://github.com/letmly/lenta-shelf-control.git /kaggle/working/lenta',
               shell=True, check=True)
from huggingface_hub import hf_hub_download
Path('/kaggle/working/lenta/_third_party/yolo_weights').mkdir(parents=True, exist_ok=True)
hf_hub_download(repo_id='openfoodfacts/price-tag-detection', filename='weights/best.pt',
                local_dir='/kaggle/working/lenta/_third_party/yolo_weights/')
print('Repo cloned (autoresearch/may16) + YOLO weights downloaded')
import os
assert os.path.exists('/kaggle/working/lenta/scripts/pipeline_v10.py'), 'pipeline_v10.py missing'
assert os.path.exists('/kaggle/working/lenta/scripts/evaluate.py'), 'evaluate.py missing'
# verify multi-frame patch applied
with open('/kaggle/working/lenta/scripts/pipeline_v10.py') as f:
    src = f.read()
assert 'MULTI-FRAME QR + 1D barcode' in src, 'pipeline_v10 not patched!'
print('Verified pipeline_v10 multi-frame barcode patch present')

In [ ]:
# === OCRWrapper + monkey-patch + run (ONE cell to preserve namespace) ===
import sys, numpy as np, cv2
from pathlib import Path

def preprocess(img):
    out = img
    if USE_DENOISE and out.ndim == 3:
        out = cv2.fastNlMeansDenoisingColored(out, None, 7, 7, 5, 11)
    if USE_CLAHE and out.ndim == 3:
        lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB); l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        out = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)
    if USE_SHARPEN:
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
        out = cv2.filter2D(out, -1, kernel)
    return out

class OCRWrapper:
    def __init__(self, engine):
        self.engine = engine
        if engine == 'rapidocr':
            from rapidocr_onnxruntime import RapidOCR
            self._ocr = RapidOCR()
        elif engine == 'easyocr':
            import easyocr
            self._ocr = easyocr.Reader(['ru','en'], gpu=False, verbose=False)
        elif engine == 'tesseract':
            import pytesseract
            self._ocr = pytesseract
        elif engine == 'qwen25_vl_3b':
            import torch
            from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
            from qwen_vl_utils import process_vision_info
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                'Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
            self.processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
            self._pv = process_vision_info
            self._torch = torch

    def __call__(self, img, **kw):
        img = preprocess(img)
        if self.engine == 'rapidocr':
            try: return self._ocr(img, **kw)
            except TypeError: return self._ocr(img)
        if self.engine == 'easyocr':
            res = self._ocr.readtext(img, detail=1, paragraph=False)
            return [[list(r[0]), r[1], float(r[2])] for r in res], 0.0
        if self.engine == 'tesseract':
            try: txt = self._ocr.image_to_string(img, lang='rus+eng')
            except: return None, 0.0
            return [[[0,0,0,0], l.strip(), 0.5] for l in txt.split('\n') if l.strip()], 0.0
        if self.engine == 'qwen25_vl_3b':
            from PIL import Image
            pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            msgs = [{'role':'user','content':[{'type':'image','image':pil},
                {'type':'text','text':'Extract the product name from this Russian price tag. Return only the product name in Russian.'}]}]
            text = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = self._pv(msgs)
            inputs = self.processor(text=[text], images=image_inputs, padding=True, return_tensors='pt').to(self.model.device)
            with self._torch.no_grad():
                out = self.model.generate(**inputs, max_new_tokens=128, do_sample=False)
            out_t = [o[len(i):] for i,o in zip(inputs.input_ids, out)]
            pred = self.processor.batch_decode(out_t, skip_special_tokens=True)[0].strip()
            return [[[0,0,0,0], pred, 1.0]], 0.0

# Run pipeline
sys.path.insert(0, '/kaggle/working/lenta/scripts')
import pipeline_v10
pipeline_v10.YOLO_WEIGHTS = Path('/kaggle/working/lenta/_third_party/yolo_weights/weights/best.pt')
# Closure captures OCRWrapper from this cell's namespace (preserved in module globals)
_wrap_engine = OCR_ENGINE
def _make_ocr(): return OCRWrapper(_wrap_engine)
pipeline_v10.RapidOCR = _make_ocr

from pipeline_v10 import process_video
OUT_CSV = Path('/kaggle/working/exp_pred.csv')
df_pred = process_video(Path(video_path), fps_sample=FPS_SAMPLE, output_csv=OUT_CSV,
                         filename_in_csv='26_12-20.mp4', upscale=UPSCALE, yolo_conf=YOLO_CONF)
print(f'pred rows: {len(df_pred) if df_pred is not None else 0}')

if DEFAULT_NET_FOR_TEXT_FIELDS and df_pred is not None:
    for f in ['product_name','additional_info','code','special_symbols','print_datetime']:
        if f in df_pred.columns:
            df_pred[f] = df_pred[f].fillna('нет').apply(lambda x: 'нет' if str(x).strip() in ('','nan','None') else x)
    df_pred.to_csv(OUT_CSV, index=False, encoding='utf-8')

In [ ]:
# === Evaluate ===
import sys
sys.path.insert(0, '/kaggle/working/lenta/scripts')
from evaluate import match_rows, compare_fields
import pandas as pd
pred_df = pd.read_csv(OUT_CSV, encoding='utf-8')
gt_df = pd.read_csv(csv_path, encoding='utf-8')
n_gt = len(gt_df); n_pred = len(pred_df)
matches = match_rows(pred_df.copy(), gt_df.copy())
scores = []
for gi, pi, _ in matches:
    if pi is None: scores.append(0.0); continue
    s, _ = compare_fields(pred_df.iloc[pi], gt_df.iloc[gi])
    scores.append(s)
matched = sum(1 for _,p,_ in matches if p is not None)
qualified = sum(1 for s in scores if s>=0.8)
mean = sum(scores)/len(scores) if scores else 0.0
print('---')
print(f'matched:     {matched}/{n_gt} ({matched/n_gt:.4f})')
print(f'mean_score:  {mean:.4f}')
print(f'qualified:   {qualified}/{n_gt} ({qualified/n_gt:.4f})')
print(f'n_pred:      {n_pred}')
print(f'main_metric: {qualified/n_gt:.6f}')
print(f'tie_break:   {mean:.6f}')
import json
with open('/kaggle/working/exp_results.json','w',encoding='utf-8') as f:
    json.dump({'description':DESCRIPTION,'main_metric':qualified/n_gt,'tie_break':mean,
               'matched':matched,'matched_pct':matched/n_gt,'qualified':qualified,
               'n_gt':n_gt,'n_pred':n_pred}, f, ensure_ascii=False, indent=1)